# Vérification de la récupération des risques (Géorisques)

Ce notebook vérifie que `download.risques.download_risques` interroge correctement l'API publique Géorisques pour une commune (liste des risques GASPAR, zone de sismicité réglementaire, potentiel radon) et écrit le résultat en JSON.

⚠️ Ces données sont **attributaires** (par commune), pas géométriques : l'API Géorisques ne fournit pas les polygones des zones à risque (zones inondables, zonage PPR...) via cette voie. Seule la limite communale est donc affichée sur la carte, à titre de contexte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary
from download.risques import download_risques

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Récupération des risques de la commune

In [ ]:
written = download_risques(entity.code_insee)
written

## 3. Contenu récupéré

In [ ]:
import json
import pandas as pd

risques = json.loads(written.read_text(encoding="utf-8"))

print("Zone de sismicite :", risques["zone_sismicite"])
print("Classe de potentiel radon :", risques["classe_potentiel_radon"])

pd.DataFrame(risques["risques_gaspar"])

## 4. Contexte géographique

Limite communale affichée pour contexte (pas de polygone de risque disponible via cette API). Backend `folium`, plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

boundary_wgs84 = fetch_commune_boundary(entity.code_insee).to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0.05, "fillColor": "red"},
    info_mode="on_click",
)
m.zoom_to_gdf(boundary_wgs84)
m